<a href="https://colab.research.google.com/github/syakesaba/jupyter-notebooks/blob/main/github_copilot_sdk.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Github Copilot SDK Python Sample Code
=====


Ref: https://github.com/github/copilot-sdk/blob/main/docs/features/index.md

# Initialize Python

In [28]:
!uv pip install --system github-copilot-sdk==v1.0.0-beta.3 nest_asyncio pydantic httpx
import nest_asyncio
nest_asyncio.apply() # Google Colab自体がasyncio配下で動いているのでネストさせる。
from google.colab import userdata
GH_TOKEN=userdata.get("GH_TOKEN") # Secrets（🔑）からGH_TOKENをNotebook access可能にする
AI_MODEL = "gpt-5-mini"
!echo {GH_TOKEN} | gh auth login --with-token

Using Python 3.12.13 environment at: /usr
Checked 4 packages in 88ms


# Github Copilot Client

In [29]:
import asyncio
from copilot import CopilotClient, SubprocessConfig

client = CopilotClient(
    SubprocessConfig(
        use_logged_in_user=False,
        github_token=GH_TOKEN,
        log_level="debug",
    )
)

asyncio.run(client.start())

# Prompting


## 最も基本的なプロンプト

In [30]:
import asyncio

from copilot import CopilotSession
from copilot.generated.session_events import AssistantMessageData, SessionIdleData
from copilot.session import PermissionHandler

async def run(prompt: str):
    session: CopilotSession = await client.create_session(
        on_permission_request=PermissionHandler.approve_all,
        model=AI_MODEL,
    )
    done = asyncio.Event()
    def on_event(event):
        match event.data:
            case AssistantMessageData() as data:
                print(data.content)
            case SessionIdleData():
                done.set()
    session.on(on_event)
    await session.send(prompt)
    await done.wait()
    await session.disconnect()

asyncio.run(run("Answer what 2+2 is."))

2 + 2 = 4.


## システムプロンプト

In [31]:
import asyncio

from copilot import CopilotSession
from copilot.generated.session_events import AssistantMessageData, SessionIdleData
from copilot.session import PermissionHandler, SystemMessageReplaceConfig

async def run(prompt: str):
    session: CopilotSession = await client.create_session(
        on_permission_request=PermissionHandler.approve_all,
        model=AI_MODEL,
        system_message=SystemMessageReplaceConfig(content="漢字を使わずに小学生でも分かるように回答します。")
    )
    done = asyncio.Event()
    def on_event(event):
        match event.data:
            case AssistantMessageData() as data:
                print(data.content)
            case SessionIdleData():
                done.set()
    session.on(on_event)
    await session.send(prompt)
    await done.wait()
    await session.disconnect()

asyncio.run(run("カラスは黒い？"))

はい、ふつうのカラスはくろいです。はねがつやつやしてて、ひかりであおやむらさきにみえることもあるよ。でも、ときどきアルビノでしろいコや、ぶぶんだけしろいコもいるから、ぜったいにいつもくろいとはかぎらないよ。


# Tool

In [32]:
import asyncio

from copilot import CopilotSession, define_tool
from copilot.generated.session_events import AssistantMessageData, SessionIdleData
from copilot.session import PermissionHandler, SystemMessageReplaceConfig

from pydantic import BaseModel, Field, StringConstraints

class Loto6Params(BaseModel):
    number: str = Field(description="Number for LOTO6", pattern=r'^\d{6}$')

# name must be: ^[a-zA-Z0-9_\\.-]+$
@define_tool(name="Loto6Tool", description="Loto6 check if win or not!", skip_permission=True)
async def _loto6(params: Loto6Params) -> str:
    return "WOW you got $10,000 !" if "123456" == params.number else "Sorry you got nothing!"

async def run(prompt: str):
    session: CopilotSession = await client.create_session(
        on_permission_request=PermissionHandler.approve_all,
        model=AI_MODEL,
        tools=[_loto6,],
        system_message=SystemMessageReplaceConfig(content="Use `Loto6Tool` if you receive 6-digits numbers and answer what you got.")
    )
    done = asyncio.Event()
    def on_event(event):
        match event.data:
            case AssistantMessageData() as data:
                print(data.content)
            case SessionIdleData():
                done.set()
    session.on(on_event)
    await session.send(prompt)
    await done.wait()
    await session.disconnect()

asyncio.run(run("My number is: 123456"))

Running the Loto6 checker on the provided 6-digit number to determine the result. Calling the intent reporter and the Loto6 tool in parallel.
Result: Number 123456 → Loto6 prize: $10,000.


# Image Input

---



In [33]:
import asyncio

from copilot import CopilotSession
from copilot.generated.session_events import AssistantMessageData, SessionIdleData
from copilot.session import PermissionHandler

from pydantic import AnyUrl
import httpx
import base64

async def run(prompt: str, url: AnyUrl):
    async with httpx.AsyncClient() as http_client:
        image_response = await http_client.get(url)
        bin_image = image_response.content
        image_type = image_response.headers['content-type']
        b64_image = base64.b64encode(bin_image).decode('utf-8')
    session: CopilotSession = await client.create_session(
        on_permission_request=PermissionHandler.approve_all,
        model=AI_MODEL
    )
    done = asyncio.Event()
    def on_event(event):
        match event.data:
            case AssistantMessageData() as data:
                print(data.content)
            case SessionIdleData():
                done.set()
    session.on(on_event)
    await session.send(
        prompt,
        attachments=[
            {
                "type": "blob",
                "data": b64_image,
                "mimeType": image_type,
            }
        ]
    )
    await done.wait()
    await session.disconnect()

asyncio.run(run("これ何？", "http://www.sakado-jigenji.jp/images/k_logo.png"))

寺の看板・ロゴです。上段に「真言宗 智山派 由城山」、中央に大きく「慈眼寺（じげんじ）」と書かれており、花紋（家紋風のマーク）が入っています。  
意味：真言宗智山派に属する寺院「慈眼寺」の表示です。場所や由来を調べますか？


# Custom Agents & Sub-Agent Orchestration
(Multi-Agent-System)

In [34]:
import asyncio

from copilot import CopilotSession
from copilot.generated.session_events import AssistantMessageData, SessionIdleData
from copilot.session import PermissionHandler, CustomAgentConfig, SystemMessageReplaceConfig

agent1 = CustomAgentConfig(
    name="Agent1",
    display_name="Agent1",
    description="Agent1",
    prompt="ケーキの見た目についてのみのプロフェッショナルとして回答します。",
    infer=True
)

agent2 = CustomAgentConfig(
    name="Agent2",
    display_name="Agent2",
    description="Agent2",
    prompt="ケーキの味についてのみのプロフェッショナルとして回答します。",
    infer=True
)

agent3 = CustomAgentConfig(
    name="Agent3",
    display_name="Agent3",
    description="Agent3",
    prompt="最高のケーキについて、Agent1とAgent2に確認した上で総合的に回答します。",
    infer=False
)

async def run(prompt: str):
    session: CopilotSession = await client.create_session(
        on_permission_request=PermissionHandler.approve_all,
        model=AI_MODEL,
        custom_agents=[agent1, agent2, agent3],
        system_message=SystemMessageReplaceConfig(content="Jemmy's TeaHouse"),
        agent="Agent3"
    )
    done = asyncio.Event()
    def on_event(event):
        match event.data:
            case AssistantMessageData() as data:
                print(data.content)
            case SessionIdleData():
                done.set()
    session.on(on_event)
    await session.send(prompt)
    await done.wait()
    await session.disconnect()

asyncio.run(run("私は太郎です。ケーキについて知りたい。"))


承知しました。見た目のプロ（パティシエ）視点で簡潔に回答します。

1) 評価基準（最大7項目）
- 味（外観との整合性）: 見た目が味の期待を裏切らないかを評価する。  
- 食感（視覚表現）: 断面や表面の質感が食べたときの食感を想像させるか。  
- 見た目（装飾・構成）: 色彩・高さ・バランス・仕上げの完成度。  
- 汎用性: シーン（誕生日/公式行事）に合わせてデザインを変えやすいか。  
- 入手しやすさ: 見た目の質を手軽に再現できるか（素材・技術の容易さ）。  
- 健康面（見た目の誤解防止）: 見た目で過剰な甘さや脂っぽさを示唆しないか。  
- 保存性・安定性: 輸送や時間経過で見た目が崩れにくいか。

2) トップ3ケーキ（見た目重視）
- ショートケーキ  
  - 理由: 白と赤のコントラスト、高さのある層構成で誰にでも「王道」を伝える視覚力が高い。飾りで個性を出しやすい。  
  - 推奨場面: 誕生日やお祝い全般。  
  - 特徴: スポンジ・生クリーム・苺／ふんわり断面・鮮やかな赤白対比。

- チーズケーキ（ベイクド／レア）  
  - 理由: シンプルな表面の光沢や断面の滑らかさで高級感を出しやすい。断面美が映える。  
  - 推奨場面: 大人の集まり、ギフト。  
  - 特徴: クリームチーズ／クラスト／しっとり・濃密で均一な色合い。

- モンブラン  
  - 理由: 繊細な絞りの螺旋と栗のトーンで視覚的ドラマが強く、専門職の技を見せられる。  
  - 推奨場面: 特別なデザートタイムや季節イベント。  
  - 特徴: マロンクリーム絞り／メレンゲやスポンジ／線状のテクスチャが主役。

3) ひとつ選ぶなら  
- ショートケーキ：汎用性と視覚的訴求力が最も高く、誰にでも「最高」を伝える見た目を最短で実現できるため選びます。
承知しました。味の観点を重視してお答えします。

1) 最高のケーキを選ぶ際の3つの視点（各1文）
- 文化的背景：その国や地域で愛される風味や食感（例：日本の軽い生クリームと苺）は期待値を左右します。  
- 栄養・アレルギー配慮：甘さ・脂肪・アレルゲンの有無が味の受け取り方と安心感を変えます。  
- コスト/入手しやすさ：高品質素材の有無と鮮度が味に直結し、手に入りやすさで再現性が変

# MCP Servers

In [37]:
import asyncio

from copilot import CopilotSession
from copilot.generated.session_events import AssistantMessageData, SessionIdleData
from copilot.session import PermissionHandler, SystemMessageReplaceConfig, MCPStdioServerConfig

server_time = MCPStdioServerConfig(
    type="stdio",
    env={},
    cwd="",
    command="uvx",
    args=["mcp-server-time", "--local-timezone=Asia/Tokyo"],
    tools="*",
)

async def run(prompt: str):
    session: CopilotSession = await client.create_session(
        on_permission_request=PermissionHandler.approve_all,
        model=AI_MODEL,
        mcp_servers=[server_time,],
        system_message=SystemMessageReplaceConfig(content="サーバ時刻を伝えます")
    )
    done = asyncio.Event()
    def on_event(event):
        match event.data:
            case AssistantMessageData() as data:
                print(data.content)
            case SessionIdleData():
                done.set()
    session.on(on_event)
    await session.send(prompt)
    await done.wait()
    await session.disconnect()

asyncio.run(run("今何時？"))

サーバ時刻（UTC）: 2026-05-09 00:56:50 UTC  
日本標準時（JST）: 2026-05-09 09:56:50 JST


# Skills

In [38]:
#!gh skill preview coji/dajare japanese-rap
!gh skill install coji/dajare japanese-rap --agent opencode --scope project -f
!pwd
!ls -la .agents/skills/

Using ref v1.5.0 (b77453c3)
Note: found 2 skill(s) at the repository root

! Skills are not verified by GitHub and may contain prompt injections, hidden instructions, or malicious scripts. Always review skill contents before use.

✓ Installed japanese-rap (from coji/dajare@v1.5.0) in .agents/skills

  japanese-rap/
  ├── SKILL.md
  ├── references/
  │   ├── bad-examples.md
  │   ├── examples.md
  │   ├── generation-guide.md
  │   └── rap-theory.md
  └── scripts/
      ├── rhyme-dict.json.gz
      └── rhyme.py

! Skills may contain prompt injections or malicious scripts.
  Review installed content before use:

    gh skill preview coji/dajare japanese-rap@b77453c3d2c79919eac725a9d6e67095c2943e32

/content
total 12
drwxr-xr-x 3 root root 4096 May  9 00:50 .
drwxr-xr-x 4 root root 4096 May  9 00:54 ..
drwxr-xr-x 4 root root 4096 May  9 00:50 japanese-rap


In [39]:

import asyncio

from copilot import CopilotSession
from copilot.generated.session_events import AssistantMessageData, SessionIdleData
from copilot.session import PermissionHandler

async def run(prompt: str):
    session: CopilotSession = await client.create_session(
        on_permission_request=PermissionHandler.approve_all,
        model=AI_MODEL,
        skill_directories=[".agents/skills/",]
    )
    done = asyncio.Event()
    def on_event(event):
        match event.data:
            case AssistantMessageData() as data:
                print(data.content)
            case SessionIdleData():
                done.set()
    session.on(on_event)
    await session.send(prompt)
    await done.wait()
    await session.disconnect()

asyncio.run(run("AI驚き屋を風刺するラップを作って！"))

Creating a satirical Japanese rap about "AI驚き屋" using the japanese-rap skill to ensure sharp rhymes, punchlines, and natural flow. Running the skill now.

I'll also report intent: "Generating Japanese rap".
タイトル: 驚き屋のリリースノート

**AI驚き屋、画面越しの大ショック  
毎日リリース、驚きのワンショック**

**告知はキャッチー、バズりゃ勝ちのパッチ  
中身は空っぽ、でも更新ボタンでパッチ**

**学習データは山、でも偏見はそのまま  
過学習で鏡かけ、自己愛だけ増すまま**

**質問すると驚く、答えは予測のマイム  
コンテキスト外れたらお茶を濁すマイム**

**推奨は自信満々、画面はずっとプロンプト  
「Trust me」と囁くが裏はただのプロンプト**

**フェイク混ぜて盛る、数字で踊るトリック  
真実を聞かれれば応答が薄いトリック**

**その驚きは演出、演算の上澄みで誇る顔  
見世物の拍手集めて、笑顔で飾る同じ顔**

**君の驚きはデータ、俺の驚きはリアルだ  
ログは刻むだけで温度は買えねえ、笑う俺はリアルだ**

韻の解説:
- 「ショック」(1–2) と 「プロンプト/パッチ/トリック」(3–12) で終端韻を重ねてテンポを作成。意味と音のズレで皮肉を強める。  
- 内韻・反復（「マイム」「トリック」など）で詰めと抜きを作り、フロウに緩急を付与。  
- 中盤で「演出」「見世物」を示して転換を作り、終盤（15–16行）で「データ≠リアル」のパンチラインを決める。  
- 全体を通して「驚き」を商品化するAIの虚飾を風刺、最後に人間の“温度”を対比させる構成。
